In [11]:
import os
import re
import pandas as pd
import numpy as np

# --- 1. CONFIGURATION ET TELECHARGEMENT NLTK ---
try:
    import nltk
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# --- 2. CHARGEMENT ET NETTOYAGE DE STRUCTURE ---
raw_data_path = '../data/raw/phishing_email.csv'

print("--- Chargement des données brutes ---")
df = pd.read_csv(raw_data_path)

# Alignment avec le nom réel de votre colonne : 'text_combined' -> 'email_text'
if 'text_combined' in df.columns:
    df.rename(columns={'text_combined': 'email_text'}, inplace=True)

# Suppression de toute ligne parasite contenant l'en-tête dupliqué
df = df[df['email_text'] != 'text_combined'].copy()
df = df[df['email_text'] != 'email_text'].copy()

# Suppression des valeurs manquantes (NaN / null)
df.dropna(subset=['email_text', 'label'], inplace=True)

# Conversion forcée du label en entier (0 ou 1)
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

# --- 3. FONCTION DE NETTOYAGE DU TEXTE (NLP) ---
stop_words = set(stopwords.words('english'))  # Mots vides en anglais

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # 1. Passage en minuscules
    text = text.lower()
    # 2. Suppression de la ponctuation, symboles et chiffres
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Tokenisation
    tokens = word_tokenize(text)
    # 4. Suppression des stopwords et des mots isolés de 1 seule lettre
    filtered_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    # 5. Reconstitution de la phrase
    return " ".join(filtered_tokens)

print("\n--- Nettoyage du texte en cours... ---")
df['email_text'] = df['email_text'].apply(clean_text)

# Supprimer les lignes dont le texte est devenu totalement vide après filtrage
df = df[df['email_text'].str.strip() != ''].copy()

# Calcul de la longueur du texte nettoyé
df['text_length'] = df['email_text'].apply(len)

# --- 4. AFFICHAGE DES RÉSULTATS ET VÉRIFICATION ---
print("\n--- Aperçu des données propres ---")
print(df[['email_text', 'label', 'text_length']].head())

print("\n--- Répartition exacte des labels ---")
print(df['label'].value_counts())

# --- 5. SAUVEGARDE DU FICHIER PROPRE ---
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'cleaned_data.csv')

# Sauvegarde des 3 colonnes finales essentielles
df_to_save = df[['email_text', 'label', 'text_length']]
df_to_save.to_csv(output_path, index=False)

print(f"\nDonnées nettoyées et sauvegardées avec succès dans : {output_path}")

--- Chargement des données brutes ---

--- Nettoyage du texte en cours... ---

--- Aperçu des données propres ---
                                          email_text  label  text_length
0  hpl nom may see attached file hplno xls hplno xls      0           49
1  nom actual vols th forwarded sabrae zajac hou ...      0          880
2  enron actuals march april estimated actuals ma...      0          109
3  hpl nom may see attached file hplno xls hplno xls      0           49
4  hpl nom june see attached file hplno xls hplno...      0           50

--- Répartition exacte des labels ---
label
1    42889
0    39595
Name: count, dtype: int64

Données nettoyées et sauvegardées avec succès dans : ../data/processed\cleaned_data.csv
